# GUÍA MAESTRA DE EXAMEN — IA25 2do Examen

Este notebook es tu hoja de ruta para el examen práctico.
Contiene los patrones que se repiten en todos los proyectos y cómo reaccionar ante cada problema.

## Índice
1. Diagrama de decisión — ¿qué proyecto me ha caído?
2. Patrón universal de regresión
3. Patrón universal de clasificación
4. Cuándo usar qué métrica
5. Problemas más comunes y sus soluciones
6. Errores de sklearn más frecuentes
7. Checklist antes de entregar


## 1. Diagrama de decisión

```
¿Qué proyecto ha caído?
│
├─ california-e2e  ──→  Regresión, dataset local, sin leakage
│                        Mejoras: ratios, log, clustering geo, RandomizedSearchCV
│
├─ king-county     ──→  Regresión con fechas, leakage temporal es el riesgo
│                        Mejoras: split temporal, TimeSeriesSplit, feature engineering fecha
│
├─ thyroid         ──→  Clasificación médica, desbalance de clases
│                        Mejoras: F2 score, class_weight=balanced, stratify en split
│
├─ ai-chat-guardrails ──→  Chatbot + LLM, arquitectura con guardrails
│                          Mejoras: pytest config, tests robustos, persistencia historial
│
├─ IES_Teis_AI_Triage ──→  Agente RAG para correos
│                          Mejoras: inbox/ multi-JSON, deduplicación, validación
│
└─ computer-vision    ──→  OpenCV + YOLO + embeddings
                           Mejoras: validar carga, calidad imagen, verificar modelos
```


## 2. Patrón Universal de Regresión

Aplica a california-e2e y king-county.
Este es el esqueleto mínimo que siempre funciona.


In [ ]:
# ═══════════════════════════════════════════════════════════
# ESQUELETO UNIVERSAL DE REGRESIÓN
# ═══════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. CARGAR DATOS
df = pd.read_csv("ruta/al/archivo.csv")
print(df.shape, df.dtypes.value_counts())

# 2. SPLIT (aleatorio o temporal según el problema)
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. IDENTIFICAR TIPOS DE COLUMNAS
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()
print("Numéricas:", len(num_cols), "| Categóricas:", cat_cols)

# 4. PIPELINE
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_cols),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor",    RandomForestRegressor(random_state=42, n_jobs=-1)),
])

# 5. ENTRENAR
model.fit(X_train, y_train)

# 6. EVALUAR (solo una vez al final)
y_pred = model.predict(X_test)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:,.0f}")
print(f"MAE:  {mae:,.0f}")
print(f"R²:   {r2:.4f}")


## 3. Patrón Universal de Clasificación

Aplica a thyroid y a cualquier problema de clasificación que caiga.


In [ ]:
# ═══════════════════════════════════════════════════════════
# ESQUELETO UNIVERSAL DE CLASIFICACIÓN
# ═══════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# 1. CARGAR DATOS
df = pd.read_csv("ruta/al/archivo.csv")

# 2. SPLIT CON STRATIFY (siempre en clasificación)
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,     # obligatorio para mantener proporción de clases
)

print("Distribución de clases:")
print(y_train.value_counts(normalize=True).round(3))

# 3. TIPOS DE COLUMNAS
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()

# 4. PIPELINE
preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_cols),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        random_state=42,
        class_weight="balanced",    # para clases desbalanceadas
        n_jobs=-1,
    )),
])

# 5. ENTRENAR Y EVALUAR
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred, zero_division=0))


## 4. Cuándo usar qué métrica

### Regresión

| Métrica | Cuándo usar | Interpretación |
|---------|-------------|----------------|
| RMSE | Siempre | Error típico en las unidades del target |
| MAE | Hay muchos outliers | Error absoluto mediano, más robusto |
| R² | Para comparar modelos | 1.0 = perfecto, 0 = inútil, <0 = peor que la media |

### Clasificación

| Métrica | Cuándo usar | Cuándo NO usar |
|---------|-------------|----------------|
| Accuracy | Clases balanceadas | Datos desbalanceados |
| F1 | Balance precision/recall | Cuando el coste de FP ≠ FN |
| **F2** | Falsos negativos son más graves (medicina) | Cuando FP y FN cuestan igual |
| Precision | FP son muy costosos | Cuando FN son costosos |
| Recall | FN son muy costosos | Cuando FP son costosos |
| ROC-AUC | Comparar modelos | Presentar al usuario final |

### Fórmulas clave
```
Precision = TP / (TP + FP)   → de todos los que dije positivo, ¿cuántos lo son?
Recall    = TP / (TP + FN)   → de todos los positivos reales, ¿cuántos detecté?
F1        = 2 × P × R / (P + R)
F2        = 5 × P × R / (4P + R)   ← recall pesa el doble
RMSE      = sqrt(mean((y - y_pred)^2))
MAE       = mean(|y - y_pred|)
R²        = 1 - SS_res / SS_tot
```


## 5. Problemas más comunes y sus soluciones

### 5.1 Data Leakage

El leakage ocurre cuando información del futuro o del test contamina el entrenamiento.
Es el error más grave y más difícil de detectar.


In [ ]:
# ═══════════════════════════════════════════
# DETECCIÓN Y PREVENCIÓN DE DATA LEAKAGE
# ═══════════════════════════════════════════

# LEAKAGE DE PREPROCESAMIENTO (MUY COMÚN)
# MALO: fit sobre todos los datos
from sklearn.preprocessing import StandardScaler
import numpy as np

X_ejemplo = np.random.randn(100, 3)
split = 80

scaler_malo = StandardScaler()
X_scaled_malo = scaler_malo.fit_transform(X_ejemplo)   # FIT sobre train+test juntos!
# El test "sabe" la media y std de todos los datos → leakage

# BIEN: fit solo sobre train, transform sobre test
scaler_bien = StandardScaler()
X_train_scaled = scaler_bien.fit_transform(X_ejemplo[:split])  # FIT solo en train
X_test_scaled  = scaler_bien.transform(X_ejemplo[split:])      # TRANSFORM sin re-fit

print("El Pipeline de sklearn hace esto automáticamente:")
print("  pipeline.fit(X_train, y_train)")
print("  pipeline.predict(X_test)  ← transforma X_test con stats de X_train")

# LEAKAGE TEMPORAL (king-county)
print("")
print("Leakage temporal: NUNCA usar train_test_split aleatorio en datos con fecha.")
print("Usar split por posición: train=df.iloc[:split_idx], test=df.iloc[split_idx:]")


### 5.2 Clases Desbalanceadas

Cuando una clase tiene muchas más muestras que las otras, el modelo aprende a ignorar las raras.


In [ ]:
# ═══════════════════════════════════════════
# MANEJO DE CLASES DESBALANCEADAS
# ═══════════════════════════════════════════

import pandas as pd
import numpy as np

# Síntoma: accuracy alta, F1/recall en clases minoritarias muy bajo
# Diagnóstico:
y_ejemplo = pd.Series(["sano"] * 920 + ["enfermo_A"] * 60 + ["enfermo_B"] * 20)
print("Distribución de clases:")
print(y_ejemplo.value_counts(normalize=True).round(3))

# Soluciones disponibles:
print("")
print("SOLUCIÓN 1: class_weight=balanced en el clasificador")
print("  → sklearn pondera cada muestra inversamente a la frecuencia de su clase")
print("  → Sin coste computacional adicional")

print("")
print("SOLUCIÓN 2: stratify=y en train_test_split")
print("  → Garantiza que la proporción de clases es igual en train y test")

print("")
print("SOLUCIÓN 3: métricas que ignoran el desbalance")
print("  → F1 macro, F2 macro, balanced_accuracy_score")

print("")
print("SOLUCIÓN 4: oversampling con SMOTE (si tienes imblearn)")
print("  from imblearn.over_sampling import SMOTE")
print("  X_res, y_res = SMOTE().fit_resample(X_train, y_train)")


### 5.3 Valores Faltantes (NaN)

Los NaN son la causa más frecuente de errores en sklearn.


In [ ]:
# ═══════════════════════════════════════════
# MANEJO DE VALORES FALTANTES
# ═══════════════════════════════════════════

import pandas as pd
import numpy as np

# 1. DIAGNÓSTICO
df_ejemplo = pd.DataFrame({
    "age":    [25, np.nan, 30, 999, np.nan],
    "income": [50000, 60000, np.nan, 70000, 80000],
    "sex":    ["M", "F", None, "F", "M"],
})

print("NaN por columna:")
print(df_ejemplo.isnull().sum())
print(f"\nPorcentaje de NaN:")
print((df_ejemplo.isnull().mean() * 100).round(1))

# 2. TRATAR VALORES IMPOSIBLES COMO NaN
df_ejemplo.loc[df_ejemplo["age"] > 150, "age"] = np.nan
print("\nDespués de tratar 999 como NaN:")
print(df_ejemplo["age"])

# 3. ESTRATEGIAS DE IMPUTACIÓN
from sklearn.impute import SimpleImputer

print("\nEstrategias disponibles:")
print("  median       → robusta a outliers (recomendada para numéricas)")
print("  mean         → sensible a outliers")
print("  most_frequent → para categóricas")
print("  constant, fill_value=0 → cuando 0 tiene significado especial")

# 4. NUNCA ELIMINAR FILAS POR NaN EN FEATURES
print("\nNo hacer: df.dropna() → pierde datos válidos")
print("Hacer: imputar dentro del pipeline")


### 5.4 Hiperparámetros — Cuándo usar GridSearch vs RandomizedSearch


In [ ]:
# ═══════════════════════════════════════════
# GRID SEARCH vs RANDOMIZED SEARCH
# ═══════════════════════════════════════════

print("GridSearchCV")
print("  → Prueba TODAS las combinaciones")
print("  → Garantiza encontrar el mejor en el grid")
print("  → Usar con: espacio pequeño (<50 combinaciones)")
print("")
print("RandomizedSearchCV")
print("  → Prueba n_iter combinaciones aleatorias")
print("  → Más rápido para espacios grandes")
print("  → Usar con: espacio grande, n_iter=10-20 en examen")

# Notación para acceder a hiperparámetros de pasos del pipeline:
print("")
print("NOTACIÓN IMPORTANTE: paso__hiperparametro")
print("  preprocessor__geo__n_clusters  ← ClusterSimilarity dentro de preprocessor")
print("  regressor__n_estimators        ← RandomForest dentro del pipeline")

from sklearn.model_selection import RandomizedSearchCV

# Ejemplo mínimo de RandomizedSearchCV
# (sustituir model por tu pipeline real)
param_ejemplo = {
    "regressor__n_estimators":     [100, 200, 300],
    "regressor__max_depth":        [None, 10, 20],
    "regressor__min_samples_leaf": [1, 3, 5],
}
print("\nEjemplo de param_grid:")
for k, v in param_ejemplo.items():
    print(f"  {k}: {v}")
print(f"Total combinaciones: {3*3*3} = 27 (con n_iter=10, solo probamos 10)")


## 6. Errores de sklearn más frecuentes


In [ ]:
# ═══════════════════════════════════════════
# ERRORES FRECUENTES Y CÓMO RESOLVERLOS
# ═══════════════════════════════════════════

errores = [
    {
        "error": "ValueError: could not convert string to float",
        "causa": "Columna categórica pasada a un modelo que espera números",
        "solucion": "Añadir OneHotEncoder en el pipeline para columnas categóricas",
    },
    {
        "error": "ValueError: Input X contains NaN",
        "causa": "El modelo recibe NaN porque no se imputó antes",
        "solucion": "Añadir SimpleImputer al pipeline antes del modelo",
    },
    {
        "error": "NotFittedError: This Pipeline is not fitted",
        "causa": "Intentar predecir sin haber llamado a .fit()",
        "solucion": "Llamar a pipeline.fit(X_train, y_train) primero",
    },
    {
        "error": "ValueError: X has N features but Pipeline expects M",
        "causa": "El test tiene columnas diferentes al train (orden o número)",
        "solucion": "Verificar que X_test tiene las mismas columnas que X_train",
    },
    {
        "error": "ColumnTransformer columnas no encontradas",
        "causa": "Nombre de columna con typo o la columna fue eliminada antes",
        "solucion": "Imprimir df.columns y verificar los nombres exactos",
    },
    {
        "error": "F2 = 0.0 en todas las clases enfermas",
        "causa": "El modelo predice siempre la clase mayoritaria",
        "solucion": "Añadir class_weight='balanced' al clasificador",
    },
    {
        "error": "MemoryError en GridSearchCV",
        "causa": "Demasiadas combinaciones × cv folds",
        "solucion": "Usar RandomizedSearchCV con n_iter=10, cv=3",
    },
]

for e in errores:
    print(f"ERROR: {e['error']}")
    print(f"  Causa:    {e['causa']}")
    print(f"  Solución: {e['solucion']}")
    print()


## 7. Checklist antes de entregar

Antes de cerrar el notebook, verifica cada punto:


In [ ]:
# ═══════════════════════════════════════════
# CHECKLIST FINAL
# ═══════════════════════════════════════════

checklist = [
    ("Split", [
        "¿El split es correcto para el tipo de datos? (aleatorio vs temporal)",
        "¿Si es clasificación, usé stratify=y?",
        "¿NO toqué X_test antes de la evaluación final?",
    ]),
    ("Pipeline", [
        "¿El fit solo ocurre sobre X_train?",
        "¿El imputador está dentro del pipeline?",
        "¿El encoder tiene handle_unknown='ignore'?",
    ]),
    ("Métricas", [
        "¿La métrica elegida es adecuada para el problema?",
        "¿Si hay desbalance, uso F1/F2 en lugar de accuracy?",
        "¿Evalué en test una sola vez al final?",
    ]),
    ("Datos", [
        "¿Eliminé columnas que memorizan (id, timestamp si no es temporal)?",
        "¿Traté valores imposibles como NaN (no eliminé filas)?",
        "¿Verifiqué que no hay leakage entre train y test?",
    ]),
    ("Desbalance", [
        "¿Si hay desbalance, usé class_weight=balanced?",
        "¿Verifiqué que el modelo no predice siempre la clase mayoritaria?",
    ]),
]

print("=" * 60)
print("CHECKLIST FINAL ANTES DE ENTREGAR")
print("=" * 60)
for categoria, items in checklist:
    print(f"\n{categoria}:")
    for item in items:
        print(f"  [ ] {item}")
print("\n¡Repasa cada punto antes de cerrar el notebook!")
